# Initialize `oil_network_metadata` schema

Typed metadata extension tables for assets in `oil_network.assets`. Each table describes one asset class (production, refineries, pipelines, terminals, …) with FK to `oil_network.assets(asset_id)`.

**Convention:** typed columns only. Anything not yet promoted stays in `oil_network.assets.attributes` JSONB. The migration direction is JSON → typed, never the other way around — so this schema must not contain JSONB columns.

**Repeatable:** the reset cell drops and recreates `oil_network_metadata`, so the notebook converges to the same state on re-run. The parent schema `oil_network` is never touched.

**Built incrementally**, one table at a time.

- **Step 1** — `metadata_production_assets`: typed details for production-class assets.

## 0. Init

In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine, text
from IPython.display import display

PG_HOST          = os.environ.get("PG_HOST",          "localhost")
PG_PORT          = os.environ.get("PG_PORT",          "5432")
PG_DB            = os.environ.get("PG_DB",            "eia_crude")
PG_USER          = os.environ.get("PG_USER",          "eia_user")
PG_PASS          = os.environ.get("PG_PASS",          "eia_password")
PG_SCHEMA        = os.environ.get("PG_SCHEMA",        "oil_network_metadata")
PG_PARENT_SCHEMA = os.environ.get("PG_PARENT_SCHEMA", "oil_network")
PG_URL = f"postgresql+psycopg2://{PG_USER}:{PG_PASS}@{PG_HOST}:{PG_PORT}/{PG_DB}"

engine = create_engine(
    PG_URL,
    connect_args={"options": f"-csearch_path={PG_SCHEMA},{PG_PARENT_SCHEMA},public"},
    future=True,
)

with engine.connect() as c:
    ver, db, usr = c.execute(text("SELECT version(), current_database(), current_user")).one()
print(f"Connected:        {db} as {usr}")
print(f"Server:           {ver.splitlines()[0]}")
print(f"Target schema:    {PG_SCHEMA}")
print(f"Parent schema:    {PG_PARENT_SCHEMA}")

Connected:        eia_crude as eia_user
Server:           PostgreSQL 18.3 on x86_64-windows, compiled by msvc-19.44.35223, 64-bit
Target schema:    oil_network_data_loader
Parent schema:    oil_network


## 1. Reset schema

Drop and recreate `oil_network_metadata` so this notebook is idempotent. Parent schema `oil_network` is **not** touched — this notebook only manages typed extension tables, never the core graph schema.

In [2]:
with engine.begin() as c:
    c.execute(text(f"DROP SCHEMA IF EXISTS {PG_SCHEMA} CASCADE"))
    c.execute(text(f"CREATE SCHEMA {PG_SCHEMA}"))

print(f"Schema `{PG_SCHEMA}` reset.")

Schema `oil_network_data_loader` reset.


## Step 1 — `metadata_production_assets`

Typed details for production-class assets (basins, sub-basins, fields, leases). One row per asset, FK'd to `oil_network.assets`. Insert is optional — assets without a metadata row are simply un-described.

**Columns**

- `asset_id` — PK; FK to `oil_network.assets(asset_id)`. `ON DELETE CASCADE` so removing an asset cleans up its metadata row.
- `production_type` — **free text for v1**. The asset graph carries a wider vocabulary than fits a tight enum (e.g. `tight_oil`, `offshore_conventional`, `conventional_plus_tight`, `niobrara_residual_plus_conventional`), so the CHECK is intentionally omitted. Promotes to FK on `production_types` once the vocabulary is canonicalised.
- `grade_label` — physical classification: `light_sweet | medium_sweet | medium_sour | heavy_sour | dilbit | condensate`. Inline CHECK; promotes to FK pattern.
- `grade_name` — commercial name (`WTI`, `WCS`, `LLS`, `Mars`, `ANS`, `Bakken`, `Eagle Ford`, …). Free text for v1; promotes to FK when `oil_network_metadata.crude_grades` lookup is added.
- `gravity_api`, `sulphur_pct` — physical quality. CHECKs reject non-physical values.
- `capacity_bd` — productive capacity in **kbd** (thousand barrels per day).
- `well_count` — number of producing wells, where known. NULL for aggregates.
- `operator` — operating company, where meaningful (mostly NULL at basin/aggregate level).
- `vintage_notes` — free text on field development era / decline status.

**No `attributes` JSONB column.** Un-promoted fields stay in `oil_network.assets.attributes` on the parent row. Migration direction is monotonic: JSON → typed, never typed → JSON.

In [3]:
with engine.begin() as c:
    c.execute(text(f"""
        CREATE TABLE {PG_SCHEMA}.metadata_production_assets (
            asset_id        TEXT    PRIMARY KEY
                            REFERENCES {PG_PARENT_SCHEMA}.assets(asset_id) ON DELETE CASCADE,
            production_type TEXT,
            grade_label     TEXT,
            grade_name      TEXT,
            gravity_api     NUMERIC,
            sulphur_pct     NUMERIC,
            capacity_bd     NUMERIC,
            well_count      INTEGER,
            operator        TEXT,
            vintage_notes   TEXT,
            CHECK (gravity_api IS NULL OR gravity_api > 0),
            CHECK (sulphur_pct IS NULL OR sulphur_pct BETWEEN 0 AND 100),
            CHECK (capacity_bd IS NULL OR capacity_bd >= 0),
            CHECK (well_count  IS NULL OR well_count  >= 0),
            CHECK (grade_label IS NULL OR grade_label IN (
                'light_sweet','medium_sweet','medium_sour',
                'heavy_sour','dilbit','condensate'
            ))
        )
    """))
    c.execute(text(f"CREATE INDEX ix_mpa_grade_name      ON {PG_SCHEMA}.metadata_production_assets(grade_name)"))
    c.execute(text(f"CREATE INDEX ix_mpa_production_type ON {PG_SCHEMA}.metadata_production_assets(production_type)"))

    c.execute(text(f"""
        COMMENT ON TABLE {PG_SCHEMA}.metadata_production_assets IS
            'Typed metadata for production-class assets. FK to oil_network.assets. '
            'Add columns here as JSONB fields get promoted; never add JSONB columns to this schema.'
    """))
    c.execute(text(f"""
        COMMENT ON COLUMN {PG_SCHEMA}.metadata_production_assets.production_type IS
            'Free text for v1. Source asset-graph vocabulary is wider than fits a tight CHECK list. '
            'Promotes to FK on production_types once canonicalised.'
    """))
    c.execute(text(f"""
        COMMENT ON COLUMN {PG_SCHEMA}.metadata_production_assets.grade_label IS
            'Physical classification (light_sweet, medium_sour, ...). See CHECK for allowed values.'
    """))
    c.execute(text(f"""
        COMMENT ON COLUMN {PG_SCHEMA}.metadata_production_assets.grade_name IS
            'Commercial name (WTI, WCS, LLS, Mars, ANS, Bakken, ...). Free text for now; '
            'lookup table oil_network_metadata.crude_grades is a future promotion.'
    """))
    c.execute(text(f"""
        COMMENT ON COLUMN {PG_SCHEMA}.metadata_production_assets.capacity_bd IS
            'Productive capacity in thousand barrels per day (kbd).'
    """))
    c.execute(text(f"""
        COMMENT ON COLUMN {PG_SCHEMA}.metadata_production_assets.vintage_notes IS
            'Free text on field vintage / development era / decline status.'
    """))

print(f"Step 1 created: {PG_SCHEMA}.metadata_production_assets.")

Step 1 created: oil_network_data_loader.metadata_production_assets.


## Step 2 — Populate from `oil_network.assets`

Migrates the only field currently encoded in the asset graph for production assets — `configuration.production_type` — out of `oil_network.assets.attributes` JSONB and into the new typed column. One row per **physical, production-class** asset; the starter graph has 12 (11 US physical + 1 Canadian aggregate).

The other typed columns (`grade_label`, `grade_name`, `gravity_api`, `sulphur_pct`, `capacity_bd`, `well_count`, `operator`, `vintage_notes`) stay NULL — there's no source for them in the JSON. Populate by hand, by a follow-up data source, or via `UPDATE` once values are available.

`ON CONFLICT (asset_id) DO UPDATE` makes the cell idempotent: re-running just refreshes existing rows without duplicating. Filter is `kind = 'physical' AND node_class = 'production'`; basin aggregates and PADD-level views are abstract assets with no `production_type` to record (they roll up via aggregation formulas in the variable layer).

In [4]:
with engine.begin() as c:
    res = c.execute(text(f"""
        INSERT INTO {PG_SCHEMA}.metadata_production_assets (asset_id, production_type)
        SELECT
            a.asset_id,
            a.attributes -> 'configuration' ->> 'production_type'
        FROM {PG_PARENT_SCHEMA}.assets a
        WHERE a.attributes ->> 'kind'       = 'physical'
          AND a.attributes ->> 'node_class' = 'production'
        ON CONFLICT (asset_id) DO UPDATE SET
            production_type = EXCLUDED.production_type
        RETURNING asset_id
    """))
    touched = [r[0] for r in res.fetchall()]

with engine.connect() as c:
    df = pd.read_sql(text(f"""
        SELECT asset_id, production_type
        FROM {PG_SCHEMA}.metadata_production_assets
        ORDER BY asset_id
    """), c)

print(f"Step 2 upsert: {len(touched)} rows touched.")
display(df)

Step 2 upsert: 19 rows touched.


,asset_id,production_type
0,alaska_north_slope,conventional_onshore_offshore
1,bakken_mt,tight_oil
2,bakken_nd,tight_oil
3,california_conventional,conventional_onshore_offshore
4,canadian_oil_sands,None
5,colorado_conventional,niobrara_residual_plus_conventional
6,eagle_ford_tx,tight_oil
7,gulf_of_america,offshore_conventional
8,montana_other,conventional
9,oklahoma_conventional,conventional_plus_tight


## Step 3 — Curated metadata for the 12 production assets

Step 2 brought `production_type` over from JSONB. Step 3 fills in the rest of the typed columns — `grade_label`, `grade_name`, `gravity_api`, `sulphur_pct`, `capacity_bd`, `operator`, `vintage_notes` — for each of the 12 physical production assets. This is **curated data**, not derived from anything in the repo:

- **`capacity_bd`** matches `coverage_check.py` (the project-canonical 2023/24 production averages, used by the coverage audit). Don't edit one without the other.
- **`gravity_api` / `sulphur_pct`** are widely-published assay specs for the marketed grade. Round to one decimal where the upstream source is itself an average.
- **`grade_name`** is the commercial/marketed name — `WTI Midland`, `Bakken`, `Mars Blend`, `WCS`, etc. Free text in v1; will become an FK to `oil_network_metadata.crude_grades` when that lookup table lands.
- **`operator`** lists the top operators by basin / play, ~2024. M&A churns this list — refresh periodically.
- **`vintage_notes`** is free text on play history, recovery method, and current trajectory.
- **`well_count`** stays NULL — no clean source at this aggregation level (Texas RRC and ND DMR publish well counts but they're not reliably summable to the basin-sub-region level used here).

`UPDATE … WHERE asset_id = …` so the cell is idempotent: re-running just refreshes the curated columns. Step 2's `production_type` is left untouched.

The total modelled capacity from the curated values should match `coverage_check.py`'s 11,690 kbd US total (Permian-TX 4,600 + Permian-NM 1,100 + Bakken-ND 1,100 + Bakken-MT 60 + Eagle Ford 1,200 + GoM 1,800 + ANS 420 + CA 290 + OK 400 + WY 270 + CO 450). Plus 4,900 kbd for `canadian_oil_sands` (full WCSB, of which ~3,700 routes to the US).

In [5]:
# Curated production-asset details. Sources:
#   - capacity_bd matches coverage_check.py (project-canonical 2023/24 averages)
#   - gravity_api / sulphur_pct: published assay specs for the marketed grade
#   - operator: top operators by basin / play, ~2024 (changes with M&A — refresh periodically)
#   - vintage_notes: free text on play history, recovery method, current trajectory
PRODUCTION_ASSET_DETAILS = [
    {
        "asset_id":      "permian_tx",
        "grade_label":   "light_sweet",
        "grade_name":    "WTI Midland",
        "gravity_api":   41.0,
        "sulphur_pct":   0.30,
        "capacity_bd":   4600,
        "well_count":    None,
        "operator":      "multiple (ExxonMobil, Chevron, Pioneer/ExxonMobil, Diamondback, Occidental, ConocoPhillips, EOG)",
        "vintage_notes": "Tight-oil play; horizontal-drilling boom from ~2010. Midland Basin (east) plus Delaware Basin Texas portion. Marketed as WTI Midland; quality-bank deductions when blending into Cushing-spec WTI.",
    },
    {
        "asset_id":      "permian_nm",
        "grade_label":   "light_sweet",
        "grade_name":    "WTI Midland",
        "gravity_api":   42.0,
        "sulphur_pct":   0.25,
        "capacity_bd":   1100,
        "well_count":    None,
        "operator":      "multiple (Devon, EOG, ConocoPhillips, Mewbourne, Permian Resources, Marathon)",
        "vintage_notes": "Delaware Basin western portion (Lea, Eddy counties). Slightly lighter / sweeter than Texas-side average. Fastest-growing Permian sub-region since ~2018.",
    },
    {
        "asset_id":      "bakken_nd",
        "grade_label":   "light_sweet",
        "grade_name":    "Bakken",
        "gravity_api":   41.0,
        "sulphur_pct":   0.18,
        "capacity_bd":   1100,
        "well_count":    None,
        "operator":      "multiple (Continental, Hess, Chord Energy, Marathon, ExxonMobil/XTO)",
        "vintage_notes": "Williston Basin tight-oil play. Horizontal + multi-stage frac boom from 2008. ~90% of Bakken total output. DAPL is the primary export route since 2017.",
    },
    {
        "asset_id":      "bakken_mt",
        "grade_label":   "light_sweet",
        "grade_name":    "Bakken",
        "gravity_api":   41.0,
        "sulphur_pct":   0.18,
        "capacity_bd":   60,
        "well_count":    None,
        "operator":      "multiple (Oasis, Continental, Kraken)",
        "vintage_notes": "Montana portion of Williston Basin — ~10% of Bakken total. Eastern flank of the play; same quality as ND-side.",
    },
    {
        "asset_id":      "eagle_ford_tx",
        "grade_label":   "light_sweet",
        "grade_name":    "Eagle Ford",
        "gravity_api":   47.0,
        "sulphur_pct":   0.10,
        "capacity_bd":   1200,
        "well_count":    None,
        "operator":      "multiple (ConocoPhillips, EOG, Marathon, Chesapeake, Devon, Pioneer/ExxonMobil)",
        "vintage_notes": "South Texas tight-oil play; three production windows (oil, wet gas/condensate, dry gas). Boom from 2010. High blended API because condensate-window volumes lift the average. Entirely within Texas.",
    },
    {
        "asset_id":      "gulf_of_america",
        "grade_label":   "medium_sour",
        "grade_name":    "Mars Blend",
        "gravity_api":   31.0,
        "sulphur_pct":   1.6,
        "capacity_bd":   1800,
        "well_count":    None,
        "operator":      "multiple (BP, Shell, Chevron, ExxonMobil, Hess, Equinor, LLOG, Murphy)",
        "vintage_notes": "Federal offshore (BSEE/BOEM-regulated). Mixed assays: Mars Blend (medium sour, deepwater majority), HOOPS, Thunder Horse, plus shelf LLS. Recent additions Vito (2023), Whale (2024), Anchor. Reported as a single EIA series.",
    },
    {
        "asset_id":      "alaska_north_slope",
        "grade_label":   "medium_sour",
        "grade_name":    "ANS",
        "gravity_api":   31.4,
        "sulphur_pct":   1.04,
        "capacity_bd":   420,
        "well_count":    None,
        "operator":      "multiple (ConocoPhillips, ExxonMobil, Hilcorp, Santos)",
        "vintage_notes": "Prudhoe Bay first oil 1977. Declining from ~2,000 kbd peak in 1988. Routes only via TAPS to Valdez (isolated subsystem). Willow project (ConocoPhillips) sanctioned 2023; first oil expected ~2029.",
    },
    {
        "asset_id":      "california_conventional",
        "grade_label":   "heavy_sour",
        "grade_name":    "California Heavy (Kern River / SJV Heavy)",
        "gravity_api":   14.0,
        "sulphur_pct":   1.0,
        "capacity_bd":   290,
        "well_count":    None,
        "operator":      "multiple (Chevron, Aera, California Resources Corp, Berry)",
        "vintage_notes": "Kern County heavy oil + offshore state-waters production. Steam-flood / cyclic-steam stimulation since 1960s. Production declining since 1986 peak. Kern River is the most heavy-sour stream; San Ardo, Midway-Sunset comparable.",
    },
    {
        "asset_id":      "oklahoma_conventional",
        "grade_label":   "light_sweet",
        "grade_name":    "Oklahoma Sweet",
        "gravity_api":   38.0,
        "sulphur_pct":   0.30,
        "capacity_bd":   400,
        "well_count":    None,
        "operator":      "multiple (Continental, Devon, Coterra, Ovintiv, Marathon)",
        "vintage_notes": "Includes SCOOP (South-Central Oklahoma Oil Province) and STACK (Sooner Trend Anadarko Canadian Kingfisher) horizontal plays from ~2014, plus Anadarko residual. Routes primarily via Cushing.",
    },
    {
        "asset_id":      "wyoming_conventional",
        "grade_label":   "light_sweet",
        "grade_name":    "Wyoming Sweet / Niobrara",
        "gravity_api":   37.0,
        "sulphur_pct":   0.30,
        "capacity_bd":   270,
        "well_count":    None,
        "operator":      "multiple (Occidental, EOG, Devon, Continental)",
        "vintage_notes": "Mix of conventional (Powder River Basin, older fields) and Niobrara horizontal. Routes via Guernsey hub then Pony Express / Platte / Express.",
    },
    {
        "asset_id":      "colorado_conventional",
        "grade_label":   "light_sweet",
        "grade_name":    "DJ Basin Niobrara",
        "gravity_api":   41.0,
        "sulphur_pct":   0.20,
        "capacity_bd":   450,
        "well_count":    None,
        "operator":      "multiple (Occidental, Chevron (PDC acquired 2023), Civitas, Bonanza Creek)",
        "vintage_notes": "DJ Basin Niobrara horizontal play, Weld County dominant. Boom from ~2010. Routes via White Cliffs / Saddlehorn pipelines to Cushing.",
    },
    {
        "asset_id":      "canadian_oil_sands",
        "grade_label":   "dilbit",
        "grade_name":    "WCS",
        "gravity_api":   20.5,
        "sulphur_pct":   3.5,
        "capacity_bd":   4900,
        "well_count":    None,
        "operator":      "multiple (Canadian Natural Resources, Cenovus, Suncor, Imperial, MEG Energy, Athabasca)",
        "vintage_notes": "WCSB oil-sands aggregate (Athabasca + Cold Lake + Peace River). Mining since 1967 (Suncor); SAGD in-situ thermal since 2000s. Total Canadian crude production ~4,900 kbd; ~3,700 kbd routes to US via 4 cross-border pipelines (Mainline-CA, Keystone, TMX, Express+Platte). WCS = Western Canadian Select bitumen-blend; pure bitumen ~9 API.",
    },
]

with engine.begin() as c:
    for r in PRODUCTION_ASSET_DETAILS:
        c.execute(text(f"""
            UPDATE {PG_SCHEMA}.metadata_production_assets SET
                grade_label   = :grade_label,
                grade_name    = :grade_name,
                gravity_api   = :gravity_api,
                sulphur_pct   = :sulphur_pct,
                capacity_bd   = :capacity_bd,
                well_count    = :well_count,
                operator      = :operator,
                vintage_notes = :vintage_notes
            WHERE asset_id = :asset_id
        """), r)

with engine.connect() as c:
    df = pd.read_sql(text(f"""
        SELECT asset_id, production_type, grade_label, grade_name,
               gravity_api, sulphur_pct, capacity_bd, operator
        FROM {PG_SCHEMA}.metadata_production_assets
        ORDER BY asset_id
    """), c)

print(f"Step 3 update: {len(PRODUCTION_ASSET_DETAILS)} rows curated.")
print(f"Total modelled capacity: {sum(r['capacity_bd'] for r in PRODUCTION_ASSET_DETAILS):,} kbd")
display(df)

Step 3 update: 12 rows curated.
Total modelled capacity: 16,590 kbd


,asset_id,production_type,grade_label,grade_name,gravity_api,sulphur_pct,capacity_bd,operator
0,alaska_north_slope,conventional_onshore_offshore,medium_sour,ANS,31.4,1.04,420.0,"multiple (ConocoPhillips, ExxonMobil, Hilcorp,..."
1,bakken_mt,tight_oil,light_sweet,Bakken,41.0,0.18,60.0,"multiple (Oasis, Continental, Kraken)"
2,bakken_nd,tight_oil,light_sweet,Bakken,41.0,0.18,1100.0,"multiple (Continental, Hess, Chord Energy, Mar..."
3,california_conventional,conventional_onshore_offshore,heavy_sour,California Heavy (Kern River / SJV Heavy),14.0,1.00,290.0,"multiple (Chevron, Aera, California Resources ..."
4,canadian_oil_sands,None,dilbit,WCS,20.5,3.50,4900.0,"multiple (Canadian Natural Resources, Cenovus,..."
5,colorado_conventional,niobrara_residual_plus_conventional,light_sweet,DJ Basin Niobrara,41.0,0.20,450.0,"multiple (Occidental, Chevron (PDC acquired 20..."
6,eagle_ford_tx,tight_oil,light_sweet,Eagle Ford,47.0,0.10,1200.0,"multiple (ConocoPhillips, EOG, Marathon, Chesa..."
7,gulf_of_america,offshore_conventional,medium_sour,Mars Blend,31.0,1.60,1800.0,"multiple (BP, Shell, Chevron, ExxonMobil, Hess..."
8,montana_other,conventional,None,None,NaN,NaN,NaN,None
9,oklahoma_conventional,conventional_plus_tight,light_sweet,Oklahoma Sweet,38.0,0.30,400.0,"multiple (Continental, Devon, Coterra, Ovintiv..."


## Verify

List tables, columns, and FKs in the new schema.

In [6]:
with engine.connect() as c:
    tables = pd.read_sql(text("""
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = :s
        ORDER BY table_name
    """), c, params={"s": PG_SCHEMA})
    columns = pd.read_sql(text("""
        SELECT table_name, column_name, data_type, is_nullable, column_default
        FROM information_schema.columns
        WHERE table_schema = :s
        ORDER BY table_name, ordinal_position
    """), c, params={"s": PG_SCHEMA})
    fks = pd.read_sql(text("""
        SELECT tc.table_name, kcu.column_name,
               ccu.table_schema AS references_schema,
               ccu.table_name   AS references_table,
               ccu.column_name  AS references_column
        FROM information_schema.table_constraints tc
        JOIN information_schema.key_column_usage kcu
          ON  tc.constraint_name   = kcu.constraint_name
          AND tc.constraint_schema = kcu.constraint_schema
        JOIN information_schema.constraint_column_usage ccu
          ON  ccu.constraint_name   = tc.constraint_name
          AND ccu.constraint_schema = tc.constraint_schema
        WHERE tc.constraint_type = 'FOREIGN KEY'
          AND tc.table_schema    = :s
        ORDER BY tc.table_name, kcu.column_name
    """), c, params={"s": PG_SCHEMA})
    counts = pd.read_sql(text(f"""
        SELECT 'metadata_production_assets'                   AS table_name,
               COUNT(*)                                       AS row_count,
               COUNT(production_type)                         AS production_type_set,
               COUNT(grade_label) + COUNT(grade_name)
                 + COUNT(gravity_api) + COUNT(sulphur_pct)
                 + COUNT(capacity_bd) + COUNT(well_count)
                 + COUNT(operator) + COUNT(vintage_notes)     AS other_columns_set
        FROM {PG_SCHEMA}.metadata_production_assets
    """), c)

print(f"Tables in {PG_SCHEMA}:")
display(tables)
print("Columns:")
display(columns)
print("Foreign keys:")
display(fks)
print("Row counts:")
display(counts)

Tables in oil_network_data_loader:


,table_name
0,metadata_production_assets


Columns:


,table_name,column_name,data_type,is_nullable,column_default
0,metadata_production_assets,asset_id,text,NO,None
1,metadata_production_assets,production_type,text,YES,None
2,metadata_production_assets,grade_label,text,YES,None
3,metadata_production_assets,grade_name,text,YES,None
4,metadata_production_assets,gravity_api,numeric,YES,None
5,metadata_production_assets,sulphur_pct,numeric,YES,None
6,metadata_production_assets,capacity_bd,numeric,YES,None
7,metadata_production_assets,well_count,integer,YES,None
8,metadata_production_assets,operator,text,YES,None
9,metadata_production_assets,vintage_notes,text,YES,None


Foreign keys:


,table_name,column_name,references_schema,references_table,references_column
0,metadata_production_assets,asset_id,oil_network,assets,asset_id


Row counts:


,table_name,row_count,production_type_set,other_columns_set
0,metadata_production_assets,19,13,84
